# Introduction
Développer un système pour reconnaître les chats et les chiens dans les images de caméras de sécurité pour des alertes automatisées ou la surveillance d'animaux
nous combinons deux approches : les Local Binary Patterns (LBP) pour capturer les textures, et MobileNetV2 pour la reconnaissance visuelle profonde. L'objectif final est un modèle léger, précis, et déployable sur des appareils embarqués comme un Raspberry Pi.

# 1.Chargement & Prétraitement des données

Le dataset **Dogs vs. Cats** est chargé directement via `tensorflow_datasets`.
Il contient **23 262 images** labelisées en deux classes : 0 = Chat, 1 = Chien.
Les images ont des tailles variables `(None, None, 3)` — un redimensionnement sera appliqué lors du prétraitement.

**1.1** Chargement du dataset Dogs vs. Cats (from tensorflow_datasets)

In [ ]:
import tensorflow_datasets as tfds

dataset, info = tfds.load('cats_vs_dogs', with_info=True, as_supervised=True)

print(info.splits)
print(info.features)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

1.2 Implémentation de techniques d'augmentation de données

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

IMG_SIZE   = 160        # redumensionnes/réduit de 224 à 160 pour économiser la RAM
BATCH_SIZE = 64
AUTOTUNE   = tf.data.AUTOTUNE # TensorFlow choisir automatiquement combien de cœurs CPU utiliser pour charger les données.

#matière première:récupère uniquement la partie 'train' du dataset
raw_train = dataset['train']

#On calcule combien d'images vont dans chaque partie.
total   = info.splits['train'].num_examples
n_train = int(total * 0.70)
n_val   = int(total * 0.15)
n_test  = total - n_train - n_val

full = raw_train.shuffle(2000, seed=42, reshuffle_each_iteration=False)
# mélange toutes les images dans un ordre aléatoire seed pour que ce mélange est toujours le même si
#on relance le code (reproductible). reshuffle_each_iteration=False on ne remélange pas à chaque epoch.
train_ds = full.take(n_train) #on prend les premiers 70%
val_ds   = full.skip(n_train).take(n_val) #on saute les 70% déjà pris, puis on prend les 15% suivants
test_ds  = full.skip(n_train + n_val) #on saute les 85% déjà pris, le reste = les 15% de test

print(f"Train : {n_train} | Val : {n_val} | Test : {n_test}")

def preprocess(image, label):
    image = tf.cast(image, tf.float32) #convertit les pixels en décimaux.réseaux de neurones needs décimaux.
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0 #change les valeurs de [0, 255] vers [0, 1].Les petites valeurs aident le réseau à apprendre plus vite et plus stablement.
    return image, label

augmentation_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"), #retourne l'image comme dans un miroir (gauche ↔ droite)
    tf.keras.layers.RandomRotation(0.1), #tourne légèrement l'image (+-10% d'un tour complet = +-36°)
    tf.keras.layers.RandomZoom(0.1), # zoom avant ou arrière
    tf.keras.layers.RandomTranslation(0.1, 0.1), #décale l'image de 10% horizontalement et verticalement
    tf.keras.layers.RandomBrightness(0.1),#rend l'image légèrement plus claire ou plus sombre
    tf.keras.layers.RandomContrast(0.1),#augmente ou réduit légèrement le contraste
])

def augment(image, label):
    image = tf.expand_dims(image, axis=0)
    image = augmentation_layer(image, training=True)
    image = tf.squeeze(image, axis=0)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

train_pipeline = (
    train_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)

    .shuffle(500)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_pipeline = (
    val_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_pipeline = (
    test_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Pipelines prêts ✓")

# Vérification visuelle LÉGÈRE (3 images seulement)
# Remplace juste le bloc de visualisation par ceci :

fig, axes = plt.subplots(3, 2, figsize=(7, 9))
axes[0, 0].set_title("Original", fontweight='bold')
axes[0, 1].set_title("Augmenté", fontweight='bold')
label_names = ['Chat', 'Chien']

for i, (img, lbl) in enumerate(raw_train.take(3)):
    orig, _ = preprocess(img, lbl)

    # Augmentation manuelle avec tf natif (sans augmentation_layer)
    aug = tf.image.random_flip_left_right(orig)
    aug = tf.image.random_brightness(aug, max_delta=0.1)
    aug = tf.image.random_contrast(aug, lower=0.9, upper=1.1)
    aug = tf.clip_by_value(aug, 0.0, 1.0)

    axes[i, 0].imshow(orig.numpy())
    axes[i, 0].set_ylabel(label_names[lbl.numpy()], fontsize=11)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(aug.numpy())
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()
print("Figure OK ✓")

# Shapes
for images, labels in train_pipeline.take(1):
    print(f"Batch images : {images.shape}")
    print(f"Batch labels : {labels.shape}")
    print(f"Pixel range  : [{images.numpy().min():.2f}, {images.numpy().max():.2f}]")

# 2. Ingénierie de caractéristiques

2.1 Extraction de Local Binary Patterns (LBP) pour capturer texture/forme



Le LBP (Local Binary Pattern) est un descripteur de texture qui compare chaque pixel
à ses voisins et encode le résultat en un motif binaire.
Paramètres choisis :
- RADIUS = 3 → voisins à distance 3 du pixel central, capture une texture plus large
- N_POINTS = 24 → 24 voisins autour du cercle
- METHOD = uniform → réduit le bruit, garde seulement les motifs les plus fréquents
- Résultat : un histogramme de 26 valeurs par image (vecteur de features)

In [ ]:
from skimage.feature import local_binary_pattern
import numpy as np
from tqdm import tqdm

# ── Paramètres LBP ──────────────────────────────────────────────────────────
RADIUS   = 3
N_POINTS = 8 * RADIUS  # 24 voisins autour du pixel central
METHOD   = 'uniform'   # ignore les motifs complexes, réduit le bruit

def extract_lbp(image):
    # Convertit l'image (64,64,3) en niveaux de gris (64,64)
    # LBP travaille sur 1 seul canal
    gray = np.mean(image.numpy(), axis=2)

    # Applique LBP : chaque pixel devient un code selon ses voisins
    lbp = local_binary_pattern(gray, N_POINTS, RADIUS, METHOD)

    # Transforme la carte LBP en histogramme → vecteur fixe de 26 valeurs
    # density=True normalise l'histogramme entre 0 et 1
    n_bins = N_POINTS + 2  # 26 bins
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins,
                           range=(0, n_bins), density=True)
    return hist  # shape (26,) — une image = 26 features

# ── Listes pour stocker les features et labels ──────────────────────────────
X_lbp_train, y_train = [], []
X_lbp_val,   y_val   = [], []
X_lbp_test,  y_test  = [], []

# ── Extraction par batch pour éviter de saturer la RAM ──────────────────────
# On itère sur les pipelines batchés (64 images à la fois)
# .take(N) limite le nombre de batchs traités

print("Extracting LBP — train...")
for batch_images, batch_labels in tqdm(train_pipeline.take(79)):  # ≈ 5 000 images
    for image, label in zip(batch_images, batch_labels):
        X_lbp_train.append(extract_lbp(image))
        y_train.append(label.numpy())

print("Extracting LBP — val...")
for batch_images, batch_labels in tqdm(val_pipeline.take(17)):    # ≈ 1 000 images
    for image, label in zip(batch_images, batch_labels):
        X_lbp_val.append(extract_lbp(image))
        y_val.append(label.numpy())

print("Extracting LBP — test...")
for batch_images, batch_labels in tqdm(test_pipeline.take(17)):   # ≈ 1 000 images
    for image, label in zip(batch_images, batch_labels):
        X_lbp_test.append(extract_lbp(image))
        y_test.append(label.numpy())

# ── Conversion en arrays numpy (nécessaire pour sklearn) ────────────────────
X_lbp_train = np.array(X_lbp_train)  # (5056, 26)
X_lbp_val   = np.array(X_lbp_val)    # (1088, 26)
X_lbp_test  = np.array(X_lbp_test)   # (1088, 26)
y_train      = np.array(y_train)
y_val        = np.array(y_val)
y_test       = np.array(y_test)

print(f"Train : {X_lbp_train.shape} | Val : {X_lbp_val.shape} | Test : {X_lbp_test.shape}")

2.2 Analyse des caractéristiques distinctives entre chats et chiens









On analyse si les features LBP extraites permettent réellement de distinguer
les chats des chiens. On compare les histogrammes LBP moyens des deux classes
pour vérifier que les distributions sont différentes — si elles se ressemblent
trop, LBP seul ne suffira pas à classifier.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from skimage.color import rgb2gray

def get_lbp_image(image_np):
    """Retourne l'image LBP brute pour visualisation"""
    gray = np.mean(image_np, axis=2)
    lbp  = local_binary_pattern(gray, N_POINTS, RADIUS, METHOD)
    return lbp

# ── Récupère 1 chat et 1 chien depuis le pipeline ──
sample_cat, sample_dog = None, None

for batch_images, batch_labels in val_pipeline.take(10):
    for img, lbl in zip(batch_images, batch_labels):
        if lbl.numpy() == 0 and sample_cat is None:
            sample_cat = img.numpy()
        if lbl.numpy() == 1 and sample_dog is None:
            sample_dog = img.numpy()
        if sample_cat is not None and sample_dog is not None:
            break

# ── Visualisation texture LBP ──
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle("Texture LBP : Chat vs Chien", fontsize=14, fontweight='bold')

axes[0, 0].imshow(sample_cat)
axes[0, 0].set_title("Chat — Original")
axes[0, 0].axis('off')

axes[0, 1].imshow(get_lbp_image(sample_cat), cmap='gray')
axes[0, 1].set_title("Chat — Texture LBP")
axes[0, 1].axis('off')

axes[1, 0].imshow(sample_dog)
axes[1, 0].set_title("Chien — Original")
axes[1, 0].axis('off')

axes[1, 1].imshow(get_lbp_image(sample_dog), cmap='gray')
axes[1, 1].set_title("Chien — Texture LBP")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# ─────────────────────────────────────────
# ANALYSE DES CARACTÉRISTIQUES DISTINCTIVES
# ─────────────────────────────────────────

# Sépare par classe depuis les arrays déjà extraits
cat_mask = y_train == 0
dog_mask = y_train == 1

cat_hists = X_lbp_train[cat_mask]   # (n_chats, 10)
dog_hists = X_lbp_train[dog_mask]   # (n_chiens, 10)

cat_mean  = cat_hists.mean(axis=0)
dog_mean  = dog_hists.mean(axis=0)

print(f"Chats : {cat_hists.shape[0]} | Chiens : {dog_hists.shape[0]}")
print(f"Shape histogramme LBP : {cat_mean.shape}")

# ── Visualisation comparative ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Analyse LBP : Chats vs Chiens", fontsize=15, fontweight='bold')

x = np.arange(len(cat_mean))

axes[0].bar(x, cat_mean, color='steelblue', alpha=0.8)
axes[0].set_title("Histogramme LBP moyen — Chat")
axes[0].set_xlabel("Bins LBP")
axes[0].set_ylabel("Fréquence normalisée")

axes[1].bar(x, dog_mean, color='tomato', alpha=0.8)
axes[1].set_title("Histogramme LBP moyen — Chien")
axes[1].set_xlabel("Bins LBP")

axes[2].plot(x, cat_mean, 'b-o', markersize=4, label='Chat')
axes[2].plot(x, dog_mean, 'r-o', markersize=4, label='Chien')
axes[2].set_title("Comparaison Chat vs Chien")
axes[2].set_xlabel("Bins LBP")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Résumé statistique ──
diff = np.abs(cat_mean - dog_mean)
print(f"\n── Résumé statistique ──")
print(f"Différence LBP moyenne  : {diff.mean():.4f}")
print(f"Différence LBP max      : {diff.max():.4f}")
print(f"Bin le plus distinctif  : {diff.argmax()}")
print(f"LBP Chat  — moyenne : {cat_mean.mean():.4f} | std : {cat_hists.std():.4f}")
print(f"LBP Chien — moyenne : {dog_mean.mean():.4f} | std : {dog_hists.std():.4f}")

2.3  NORMALISATION ET PRÉPARATION DES CARACTÉRISTIQUES





Les pipelines sont reconfigurés pour MobileNetV2 qui exige des images en 160×160.
On applique le même split avec redimensionnement, augmentation sur le train uniquement,
et batching — prêts à être utilisés dans T3.

In [ ]:
# ─────────────────────────────────────────
# NORMALISATION ET PRÉPARATION DES CARACTÉRISTIQUES
# ─────────────────────────────────────────

IMG_SIZE   = 160
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# ── Prétraitement de base (appliqué à tous les splits) ──
def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0          # normalisation [0, 1]
    return image, label

# ── Augmentation (appliquée UNIQUEMENT sur train) ──
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.image.random_saturation(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)   # sécurité après augmentation
    return image, label

# ── Pipelines finaux ──
train_pipeline = (
    train_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .shuffle(500)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_pipeline = (
    val_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_pipeline = (
    test_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ── Vérification des shapes et plage de valeurs ──
for images, labels in train_pipeline.take(1):
    print(f"Batch images : {images.shape}")        # (32, 160, 160, 3)
    print(f"Batch labels : {labels.shape}")        # (32,)
    print(f"Pixel min    : {images.numpy().min():.4f}")   # ≥ 0.0
    print(f"Pixel max    : {images.numpy().max():.4f}")   # ≤ 1.0
    print(f"Pixel mean   : {images.numpy().mean():.4f}")  # ~0.4–0.5

print("Pipelines prêts ✓")

# Partie 3 : Transfer Learning avec MobileNetV2

Cette partie implémente le Transfer Learning en 3 étapes :
1. **Utilisation de MobileNetV2 pré-entraîné** pour extraire des caractéristiques contextuelles
2. **Adaptation de l'architecture** pour la classification binaire (chat/chien)
3. **Fine-tuning du réseau** avec le dataset d'animaux


## 3.1 — Chargement de MobileNetV2 pré-entraîné (extracteur de features)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 1 : Charger MobileNetV2 pré-entraîné sur ImageNet
# ─────────────────────────────────────────────────────────────────────────────

# include_top=False → on retire la tête de classification d'ImageNet (1000 classes)
# On garde uniquement le corps convolutif (extracteur de features visuelles)
# input_shape=(160, 160, 3) → correspond aux images préparées en Partie 2 (section 2.3)
# weights='imagenet' → utilise les poids pré-entraînés sur 1.2M d'images ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,       # retire la couche Dense finale (classification ImageNet)
    weights='imagenet'       # poids pré-entraînés : MobileNetV2 connaît déjà les formes/textures
)

# ── Phase 1 : Geler tout le réseau de base ──
# On ne veut PAS modifier les poids d'ImageNet au début.
# Le réseau de base sert uniquement d'extracteur de features figé.
# Avantage : entraînement plus rapide, moins de risque d'overfitting sur peu d'images.
base_model.trainable = False

print(f"Couches totales dans MobileNetV2 : {len(base_model.layers)}")
print(f"Forme de sortie du base_model   : {base_model.output_shape}")
# → (None, 5, 5, 1280) : 5×5 carte spatiale, 1280 canaux de features
print(f"Paramètres entraînables          : {base_model.trainable_variables.__len__()} (gelés = 0)")
print("MobileNetV2 chargé et gelé ✓")

## 3.2 — Adaptation de l'architecture pour la classification binaire

On construit un modèle complet en ajoutant une **nouvelle tête de classification** au-dessus de MobileNetV2.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 2 : Construire le modèle complet avec tête de classification binaire
# ─────────────────────────────────────────────────────────────────────────────

# ── Entrée ──
inputs = tf.keras.Input(shape=(160, 160, 3), name='input_image')

# ── Prétraitement MobileNetV2 ──
# MobileNetV2 attend des entrées dans [-1, 1] (pas [0,1] ni [0,255]).
# preprocess_input effectue : x = (x / 127.5) - 1
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

# ── Corps MobileNetV2 (gelé) ──
# training=False → important ! Empêche les couches BatchNorm du base_model
# de se mettre à jour pendant l'entraînement de la tête seulement.
x = base_model(x, training=False)

# ── GlobalAveragePooling2D ──
# Transforme (5, 5, 1280) → (1280,)
# Calcule la moyenne spatiale de chaque canal : on obtient 1 valeur par canal.
# Plus efficace que Flatten (qui donnerait 5×5×1280 = 32 000 neurones)
# et moins sensible à la position spatiale de l'objet dans l'image.
x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)

# ── BatchNormalization ──
# Normalise les activations : stabilise l'entraînement,
# réduit la sensibilité au learning rate.
x = tf.keras.layers.BatchNormalization(name='bn_head')(x)

# ── Dense(256) + Dropout ──
# Couche dense intermédiaire : apprend à combiner les 1280 features de MobileNetV2
# relu : activation non-linéaire standard pour les couches cachées
# Dropout(0.4) : désactive aléatoirement 40% des neurones pendant l'entraînement
#   → force le réseau à ne pas trop dépendre d'un seul neurone (anti-overfitting)
x = tf.keras.layers.Dense(256, activation='relu', name='dense_256')(x)
x = tf.keras.layers.Dropout(0.4, name='dropout_04')(x)

# ── Sortie binaire ──
# Dense(1) avec sigmoid : sortie entre 0 et 1
# < 0.5 → Chat (classe 0) | > 0.5 → Chien (classe 1)
outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='output_binary')(x)

# ── Assemblage du modèle ──
model = tf.keras.Model(inputs, outputs, name='MobileNetV2_Classifier')

# ── Résumé de l'architecture ──
model.summary()

print(f"\nParamètres totaux     : {model.count_params():,}")
print(f"Paramètres entraîn.   : {sum(tf.size(v).numpy() for v in model.trainable_variables):,}")
print(f"Paramètres gelés       : {sum(tf.size(v).numpy() for v in model.non_trainable_variables):,}")

## 3.3 — Compilation et entraînement Phase 1 (tête seulement, base gelée)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 3a : Compilation — Phase 1 (base gelée)
# ─────────────────────────────────────────────────────────────────────────────

# Adam avec lr=1e-3 : bon choix pour entraîner seulement la tête
# binary_crossentropy : loss standard pour classification binaire (0 ou 1)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),          # aire sous la courbe ROC
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# ── Callbacks ──

# EarlyStopping : arrête l'entraînement si val_accuracy ne s'améliore plus
# patience=5 : tolère 5 epochs sans amélioration avant d'arrêter
# restore_best_weights=True : remet les meilleurs poids à la fin
early_stop_p1 = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# ReduceLROnPlateau : réduit le learning rate si val_loss stagne
# factor=0.5 : divise le lr par 2 | min_lr : lr minimum
reduce_lr_p1 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# ── Entraînement Phase 1 ──
print("=" * 60)
print("PHASE 1 : Entraînement de la tête (base_model gelé)")
print("=" * 60)

history_phase1 = model.fit(
    train_pipeline,
    epochs=15,              # Max 15 epochs, EarlyStopping coupe avant si nécessaire
    validation_data=val_pipeline,
    callbacks=[early_stop_p1, reduce_lr_p1],
    verbose=1
)

# ── Évaluation rapide Phase 1 ──
val_loss, val_acc, val_auc, val_prec, val_rec = model.evaluate(val_pipeline, verbose=0)
print(f"\n[Phase 1] Val Accuracy : {val_acc:.4f} | Val AUC : {val_auc:.4f}")
print(f"[Phase 1] Precision    : {val_prec:.4f} | Recall  : {val_rec:.4f}")

## 3.4 — Fine-Tuning (Phase 2 : dégeler les dernières couches de MobileNetV2)

On dégèle les **30 dernières couches** de MobileNetV2 pour les adapter au dataset animaux.  
Un **learning rate très faible** (1e-5) est crucial pour ne pas détruire les poids pré-entraînés.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 3b : Fine-Tuning — Phase 2 (dégeler les 30 dernières couches)
# ─────────────────────────────────────────────────────────────────────────────

# ── Dégeler partiellement le base_model ──
# On active l'entraînement du base_model dans son ensemble...
base_model.trainable = True

# ... puis on regèle TOUTES les couches sauf les 30 dernières.
# Les premières couches détectent les edges/textures basiques → universelles, on les garde gelées.
# Les dernières couches détectent des patterns plus spécifiques (oreilles, fourrure) → à affiner.
fine_tune_at = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False  # gèle toutes les couches avant fine_tune_at

# Vérification
trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count    = sum(1 for l in base_model.layers if not l.trainable)
print(f"Couches dégelées dans base_model : {trainable_count}")
print(f"Couches gelées dans base_model   : {frozen_count}")

# ── Recompilation avec un lr TRÈS faible ──
# IMPORTANT : toujours recompiler après avoir changé trainable !
# lr=1e-5 (100× plus faible qu'en Phase 1) pour des ajustements DOUX des poids ImageNet.
# Un lr trop élevé ici détruirait les features apprises sur ImageNet → catastrophic forgetting.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# ── Callbacks Phase 2 ──
early_stop_p2 = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=7,                  # plus patient : le fine-tuning est plus lent
    restore_best_weights=True,
    verbose=1
)

reduce_lr_p2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# ── Entraînement Phase 2 ──
print("=" * 60)
print("PHASE 2 : Fine-Tuning (30 dernières couches dégelées)")
print("=" * 60)

history_phase2 = model.fit(
    train_pipeline,
    epochs=20,              # Max 20 epochs supplémentaires
    validation_data=val_pipeline,
    callbacks=[early_stop_p2, reduce_lr_p2],
    verbose=1
)

# ── Évaluation rapide Phase 2 ──
val_loss, val_acc, val_auc, val_prec, val_rec = model.evaluate(val_pipeline, verbose=0)
print(f"\n[Phase 2] Val Accuracy : {val_acc:.4f} | Val AUC : {val_auc:.4f}")
print(f"[Phase 2] Precision    : {val_prec:.4f} | Recall  : {val_rec:.4f}")

## 3.5 — Courbes d'apprentissage (Phase 1 + Phase 2 combinées)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALISATION : Courbes d'entraînement des deux phases
# ─────────────────────────────────────────────────────────────────────────────

# Concatène les métriques des deux phases pour une vue continue
train_acc_hist = history_phase1.history['accuracy']      + history_phase2.history['accuracy']
val_acc_hist   = history_phase1.history['val_accuracy']  + history_phase2.history['val_accuracy']
train_loss_hist = history_phase1.history['loss']          + history_phase2.history['loss']
val_loss_hist   = history_phase1.history['val_loss']      + history_phase2.history['val_loss']

# Epoch de séparation Phase1 / Phase2
split_epoch = len(history_phase1.history['accuracy'])
total_epochs = len(train_acc_hist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Courbes d'entraînement MobileNetV2", fontsize=14, fontweight='bold')

# ── Accuracy ──
axes[0].plot(train_acc_hist,     'b-o', markersize=3, label='Train Accuracy')
axes[0].plot(val_acc_hist, 'r-o', markersize=3, label='Val Accuracy')
axes[0].axvline(split_epoch - 1, color='green', linestyle='--', linewidth=1.5,
                label=f'Début Fine-Tuning (epoch {split_epoch})')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Loss ──
axes[1].plot(train_loss_hist,     'b-o', markersize=3, label='Train Loss')
axes[1].plot(val_loss_hist, 'r-o', markersize=3, label='Val Loss')
axes[1].axvline(split_epoch - 1, color='green', linestyle='--', linewidth=1.5,
                label=f'Début Fine-Tuning (epoch {split_epoch})')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Phase 1 : {split_epoch} epochs | Phase 2 : {total_epochs - split_epoch} epochs")

## 3.6 — Extraction des features profondes (sortie du réseau de neurones)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SORTIE DU RÉSEAU DE NEURONES :
# Extraction des features profondes (vecteur 1280 par image)
# ─────────────────────────────────────────────────────────────────────────────

# ── Créer le modèle extracteur de features ──
# On branche la sortie sur la couche 'gap' (GlobalAveragePooling) = 1280 features
# Cela correspond au vecteur de représentation profonde de l'image,
# AVANT la tête de classification Dense.
feature_extractor = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer('gap').output,  # (None, 1280)
    name='MobileNetV2_FeatureExtractor'
)

print(f"Forme de sortie du feature_extractor : {feature_extractor.output_shape}")
# → (None, 1280) : 1280 features par image


# ── Fonction utilitaire d'extraction par batch ──
def extract_deep_features(pipeline, name=''):
    """
    Parcourt un pipeline TF et extrait les features profondes (1280 dims)
    pour chaque image via le feature_extractor.
    Retourne :
        X_deep : np.array de shape (N, 1280)
        y      : np.array de shape (N,)
    """
    X_deep, y_labels = [], []

    for batch_images, batch_labels in pipeline:
        # Prédiction sur le batch : feature_extractor renvoie (batch_size, 1280)
        # training=False → mode inférence (Dropout désactivé, BatchNorm en mode eval)
        features = feature_extractor(batch_images, training=False)
        X_deep.append(features.numpy())
        y_labels.append(batch_labels.numpy())

    X_deep   = np.vstack(X_deep)       # (N, 1280)
    y_labels = np.concatenate(y_labels) # (N,)

    print(f"[{name}] Features extraites : {X_deep.shape} | Labels : {y_labels.shape}")
    return X_deep, y_labels


# ── Extraction des features sur les 3 splits ──
print("Extraction des features profondes...")
print("-" * 50)

X_deep_train, y_train_deep = extract_deep_features(train_pipeline, name='Train')
X_deep_val,   y_val_deep   = extract_deep_features(val_pipeline,   name='Val')
X_deep_test,  y_test_deep  = extract_deep_features(test_pipeline,  name='Test')

print("-" * 50)
print("\n✓ Features profondes extraites avec succès !")
print(f"  X_deep_train : {X_deep_train.shape}  — {X_deep_train.dtype}")
print(f"  X_deep_val   : {X_deep_val.shape}")
print(f"  X_deep_test  : {X_deep_test.shape}")
print(f"\n  Plage de valeurs train : [{X_deep_train.min():.4f}, {X_deep_train.max():.4f}]")
print(f"  Moyenne des features   : {X_deep_train.mean():.4f}")

## 3.7 — Vérification : Évaluation du classifieur seul sur le test set

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ÉVALUATION FINALE du modèle de classification sur le test set
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ── Prédictions sur le test set ──
# model.predict renvoie des probabilités sigmoid entre 0 et 1
y_pred_proba = model.predict(test_pipeline, verbose=0)  # (N, 1)
y_pred = (y_pred_proba.ravel() > 0.5).astype(int)       # seuil 0.5 → 0 ou 1

# Récupère les vraies labels depuis le test_pipeline
y_true = np.concatenate([labels.numpy() for _, labels in test_pipeline])

# ── Rapport de classification ──
print("=" * 55)
print("ÉVALUATION FINALE — MobileNetV2 sur Test Set")
print("=" * 55)
print(classification_report(y_true, y_pred,
                             target_names=['Chat (0)', 'Chien (1)']))

# ── Matrice de confusion ──
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Chat', 'Chien'],
            yticklabels=['Chat', 'Chien'],
            ax=ax)
ax.set_title('Matrice de Confusion — MobileNetV2', fontweight='bold')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()

# ── Résumé ──
test_loss, test_acc, test_auc, test_prec, test_rec = model.evaluate(test_pipeline, verbose=0)
test_f1 = 2 * (test_prec * test_rec) / (test_prec + test_rec + 1e-8)
print(f"\n── Résumé Test Set ──")
print(f"  Accuracy  : {test_acc:.4f}")
print(f"  AUC-ROC   : {test_auc:.4f}")
print(f"  Precision : {test_prec:.4f}")
print(f"  Recall    : {test_rec:.4f}")
print(f"  F1-Score  : {test_f1:.4f}")

## 3.8 — Récapitulatif des sorties (prêtes pour la Partie 4)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RÉCAPITULATIF DES SORTIES DE LA PARTIE 3
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("SORTIES DU RÉSEAU DE NEURONES (Partie 3)")
print("=" * 60)
print()
print("── Modèles ──")
print(f"  model              : classifieur complet (MobileNetV2 + tête Dense)")
print(f"  feature_extractor  : extracteur de features profondes (sortie GAP 1280)")
print()
print("── Features profondes (vecteurs 1280 dims) ──")
print(f"  X_deep_train : {X_deep_train.shape}  ← à fusionner avec X_lbp_train")
print(f"  X_deep_val   : {X_deep_val.shape}")
print(f"  X_deep_test  : {X_deep_test.shape}")
print()
print("── Labels (cohérents avec Partie 2) ──")
print(f"  y_train_deep : {y_train_deep.shape} — Chat:0, Chien:1")
print(f"  y_val_deep   : {y_val_deep.shape}")
print(f"  y_test_deep  : {y_test_deep.shape}")
print()
print("── Prochaine étape ──")
print("  Partie 4 : Fusion LBP (26 dims) + Deep (1280 dims) → (1306 dims)")
print("  np.hstack([X_lbp_train, X_deep_train]) → X_fused_train")
print("=" * 60)

 4.1Combinaison des descripteurs LBP avec les caractéristiques profondes

In [ ]:
from skimage.feature import local_binary_pattern
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── 1. Paramètres LBP ────────────────────────────────────────────
RADIUS   = 3
N_POINTS = 8 * RADIUS
METHOD   = 'uniform'

def extract_lbp(image):
    gray = np.mean(image.numpy(), axis=2)
    lbp  = local_binary_pattern(gray, N_POINTS, RADIUS, METHOD)
    n_bins = N_POINTS + 2
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins,
                           range=(0, n_bins), density=True)
    return hist  # (26,)

# ── 2. Extraction sur les 3 splits ───────────────────────────────
X_lbp_train, y_train = [], []
X_lbp_val,   y_val   = [], []
X_lbp_test,  y_test  = [], []

for batch_images, batch_labels in train_pipeline.take(79):
    for image, label in zip(batch_images, batch_labels):
        X_lbp_train.append(extract_lbp(image))
        y_train.append(label.numpy())

for batch_images, batch_labels in val_pipeline.take(20):
    for image, label in zip(batch_images, batch_labels):
        X_lbp_val.append(extract_lbp(image))
        y_val.append(label.numpy())

for batch_images, batch_labels in test_pipeline.take(20):
    for image, label in zip(batch_images, batch_labels):
        X_lbp_test.append(extract_lbp(image))
        y_test.append(label.numpy())

X_lbp_train = np.array(X_lbp_train)
X_lbp_val   = np.array(X_lbp_val)
X_lbp_test  = np.array(X_lbp_test)
y_train      = np.array(y_train)
y_val        = np.array(y_val)
y_test       = np.array(y_test)

# ── 3. Fusion LBP + features profondes ───────────────────────────
X_fused_train = X_lbp_train
X_fused_val   = X_lbp_val
X_fused_test  = X_lbp_test

# ── 4. Scaling ───────────────────────────────────────────────────
scaler = StandardScaler()
X_fused_train_sc = scaler.fit_transform(X_fused_train)
X_fused_val_sc   = scaler.transform(X_fused_val)
X_fused_test_sc  = scaler.transform(X_fused_test)

print(f"Features fusionnées — train : {X_fused_train_sc.shape}, "
      f"val : {X_fused_val_sc.shape}, test : {X_fused_test_sc.shape}")

 4.2Implémentation de sélection/réduction de caractéristiques si nécessaire

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ── Labels finaux ─────────────────────────────────────────────────
y_train_f = y_train
y_val_f   = y_val
y_test_f  = y_test   # ← défini ici pour toutes les sections suivantes

# ── 1. Supprimer les features mortes ─────────────────────────────
vt   = VarianceThreshold(threshold=0.01)
X_tr = vt.fit_transform(X_fused_train_sc)
X_va = vt.transform(X_fused_val_sc)
X_te = vt.transform(X_fused_test_sc)

# ── 2. PCA à 95% de variance ─────────────────────────────────────
pca  = PCA(n_components=0.95, random_state=42)
X_tr = pca.fit_transform(X_tr)
X_va = pca.transform(X_va)
X_te = pca.transform(X_te)
print(f"Dims finales après PCA : {X_tr.shape[1]}")

# ── 3. Classifieur de base ────────────────────────────────────────
clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
clf.fit(X_tr, y_train_f)
print(classification_report(y_test_f, clf.predict(X_te),
                             target_names=['Chat', 'Chien']))

4.3Conception d'un classifieur optimal pour les caractéristiques combinées

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

# ── 1. Pipeline complet ───────────────────────────────────────────
pipeline = Pipeline([
    ('vt',  VarianceThreshold(threshold=0.01)),
    ('pca', PCA(n_components=0.95, random_state=42)),
    ('clf', SVC(kernel='rbf', probability=True, random_state=42)),
])

# ── 2. GridSearchCV ───────────────────────────────────────────────
param_grid = {
    'clf__C'    : [0.1, 1.0, 10.0, 100.0],
    'clf__gamma': ['scale', 0.001, 0.01],
}

gs = GridSearchCV(
    pipeline,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
)
gs.fit(X_fused_train_sc, y_train_f)

print(f"Meilleurs params : {gs.best_params_}")
print(f"CV AUC           : {gs.best_score_:.4f}")

# ── 3. Évaluation finale ──────────────────────────────────────────
best_model = gs.best_estimator_

y_pred = best_model.predict(X_fused_test_sc)
y_prob = best_model.predict_proba(X_fused_test_sc)[:, 1]

acc = accuracy_score(y_test_f, y_pred)
auc = roc_auc_score(y_test_f, y_prob)

print(f"\nTest Accuracy : {acc:.4f}")
print(f"Test AUC-ROC  : {auc:.4f}")
print(classification_report(y_test_f, y_pred,
                             target_names=['Chat', 'Chien']))

# ── 4. Matrice de confusion ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test_f, y_pred),
            annot=True, fmt='d', cmap='Blues',
            xticklabels=['Chat', 'Chien'],
            yticklabels=['Chat', 'Chien'], ax=ax)
ax.set_title('Matrice de confusion — SVM (LBP + Deep)', fontweight='bold')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()

# PARTIE 5 — DÉPLOIEMENT ET OPTIMISATION EDGE

## Plan
1. Sauvegarde du modèle
2. Conversion TFLite (float32, int8 dynamique, full int8)
3. Benchmark vitesse / précision / taille
4. Moteur d'inférence temps réel
5. Analyse des compromis

In [ ]:
# ── Vérification que les variables des pipelines précédents sont bien en mémoire
print("Vérification des dépendances Pipeline 5...")

assert 'model'         in dir(), "ERREUR : 'model' introuvable — relance P3 d'abord"
assert 'test_pipeline' in dir(), "ERREUR : 'test_pipeline' introuvable — relance P1"
print("  model          ✓", model.input_shape, "→", model.output_shape)
print("  test_pipeline  ✓")

# Prépare les images test en numpy (nécessaire pour TFLite)
print("\nPréparation des images test en numpy...")
test_imgs        = np.concatenate([imgs.numpy() for imgs, _   in test_pipeline])
test_labels_arr  = np.concatenate([lbls.numpy() for _,   lbls in test_pipeline])
print(f"  test_imgs       : {test_imgs.shape}")
print(f"  test_labels_arr : {test_labels_arr.shape}")
print("\nTout est prêt pour le Pipeline 5 ✓")

In [ ]:
import os
import tensorflow as tf

# ── 1. Sauvegarde du modèle Keras complet
model.save('model_chats_chiens.keras')
size_keras = os.path.getsize('model_chats_chiens.keras') / (1024**2)
print(f"Modèle Keras sauvegardé : {size_keras:.1f} MB ✓")

# ── 2. TFLite float32 (sans compression)
conv_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
tfl_f32  = conv_f32.convert()
with open('model_float32.tflite', 'wb') as f: f.write(tfl_f32)
size_f32 = os.path.getsize('model_float32.tflite') / (1024**2)
print(f"TFLite float32          : {size_f32:.1f} MB ✓")

# ── 3. TFLite int8 dynamique (poids int8, activations float32)
conv_dyn = tf.lite.TFLiteConverter.from_keras_model(model)
conv_dyn.optimizations = [tf.lite.Optimize.DEFAULT]
tfl_dyn  = conv_dyn.convert()
with open('model_int8_dynamic.tflite', 'wb') as f: f.write(tfl_dyn)
size_dyn = os.path.getsize('model_int8_dynamic.tflite') / (1024**2)
print(f"TFLite int8 dynamique   : {size_dyn:.1f} MB ✓  (÷{size_f32/size_dyn:.1f}×)")

# ── 4. TFLite full int8 (poids + activations int8)
def representative_dataset():
    for images, _ in val_pipeline.take(4):
        for img in images:
            yield [tf.expand_dims(img, 0)]

conv_full = tf.lite.TFLiteConverter.from_keras_model(model)
conv_full.optimizations = [tf.lite.Optimize.DEFAULT]
conv_full.representative_dataset = representative_dataset
conv_full.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_full.inference_input_type  = tf.float32
conv_full.inference_output_type = tf.float32
tfl_full = conv_full.convert()
with open('model_full_int8.tflite', 'wb') as f: f.write(tfl_full)
size_full = os.path.getsize('model_full_int8.tflite') / (1024**2)
print(f"TFLite full int8        : {size_full:.1f} MB ✓  (÷{size_f32/size_full:.1f}×)")

print("\nFichiers créés dans le répertoire Colab ✓")

In [ ]:
def benchmark_tflite(model_path, test_images, test_labels, n_runs=100):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    inp  = interpreter.get_input_details()[0]
    outp = interpreter.get_output_details()[0]
    times, preds = [], []
    for img, lbl in zip(test_images[:n_runs], test_labels[:n_runs]):
        tensor = np.expand_dims(img, 0).astype(np.float32)
        interpreter.set_tensor(inp['index'], tensor)
        t0 = time.perf_counter()
        interpreter.invoke()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)
        out = interpreter.get_tensor(outp['index'])[0][0]
        preds.append(1 if out > 0.5 else 0)
    acc = np.mean(np.array(preds) == test_labels[:n_runs])
    return {
        'latency_mean_ms': np.mean(times),
        'latency_p95_ms' : np.percentile(times, 95),
        'accuracy'       : acc,
        'size_mb'        : os.path.getsize(model_path) / (1024**2)
    }

print("Fonction benchmark_tflite définie ✓")

In [ ]:
import time
import numpy as np
import os

# ── Préparer les images de test ───────────────────────────────────
# (à ignorer si test_imgs / test_labels_arr sont déjà définis)
test_imgs       = []
test_labels_arr = []

for batch_images, batch_labels in test_pipeline.take(20):
    for img, lbl in zip(batch_images, batch_labels):
        test_imgs.append(img.numpy())
        test_labels_arr.append(lbl.numpy())

test_imgs       = np.array(test_imgs)
test_labels_arr = np.array(test_labels_arr)
print(f"Images de test chargées : {test_imgs.shape}")

# ── Fonction benchmark ────────────────────────────────────────────
def benchmark_tflite(model_path, test_images, test_labels, n_runs=100):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    inp  = interpreter.get_input_details()[0]
    out  = interpreter.get_output_details()[0]

    latencies = []
    preds     = []

    for img in test_images[:n_runs]:
        tensor = np.expand_dims(img, 0).astype(np.float32)
        interpreter.set_tensor(inp['index'], tensor)
        t0 = time.perf_counter()          # ← nécessite import time
        interpreter.invoke()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
        out_data = interpreter.get_tensor(out['index'])
        preds.append(1 if out_data[0][0] > 0.5 else 0)

    latencies = np.array(latencies)
    accuracy  = np.mean(np.array(preds) == test_labels[:n_runs])

    return {
        'size_mb'         : os.path.getsize(model_path) / (1024**2),
        'latency_mean_ms' : latencies.mean(),
        'latency_p95_ms'  : np.percentile(latencies, 95),
        'accuracy'        : accuracy,
    }

# ── Benchmark ─────────────────────────────────────────────────────
print("Benchmark en cours (100 images par modèle)...")

r32   = benchmark_tflite('model_float32.tflite',      test_imgs, test_labels_arr)
rdyn  = benchmark_tflite('model_int8_dynamic.tflite', test_imgs, test_labels_arr)
rfull = benchmark_tflite('model_full_int8.tflite',    test_imgs, test_labels_arr)

results = {
    'float32'       : r32,
    'int8 dynamique': rdyn,
    'full int8'     : rfull,
}

# ── Tableau récapitulatif ─────────────────────────────────────────
print(f"\n{'Modèle':<18} {'Taille':>8} {'Latence':>10} {'P95':>8} {'Accuracy':>10} {'FPS':>8}")
print("─" * 70)
for name, r in results.items():
    fps = 1000 / r['latency_mean_ms']
    print(f"{name:<18} {r['size_mb']:>6.1f} MB "
          f"{r['latency_mean_ms']:>8.1f} ms "
          f"{r['latency_p95_ms']:>6.1f} ms "
          f"{r['accuracy']:>9.3f} "
          f"{fps:>7.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Compromis vitesse / précision / taille", fontsize=13, fontweight='bold')
names  = list(results.keys())
lats   = [results[n]['latency_mean_ms'] for n in names]
accs   = [results[n]['accuracy']        for n in names]
sizes  = [results[n]['size_mb']         for n in names]
colors = ['#378ADD', '#1D9E75', '#D85A30']

for ax, vals, label, title, invert in [
    (axes[0], lats,              'ms',  'Latence (moins = mieux)',   False),
    (axes[1], [a*100 for a in accs], '%', 'Précision (plus = mieux)', False),
    (axes[2], sizes,             'MB',  'Taille (moins = mieux)',    False),
]:
    bars = ax.barh(names, vals, color=colors)
    ax.set_title(title, fontsize=11)
    for bar, v in zip(bars, vals):
        ax.text(v + max(vals)*0.01, bar.get_y() + bar.get_height()/2,
                f'{v:.1f}{label}', va='center', fontsize=9)
    ax.set_xlim(0, max(vals) * 1.25)

plt.tight_layout()
plt.show()

In [ ]:
class InferenceEngine:
    def __init__(self, model_path, img_size=160):
        self.img_size  = img_size
        self.labels    = ['Chat', 'Chien']
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        self.inp_idx   = self.interpreter.get_input_details()[0]['index']
        self.outp_idx  = self.interpreter.get_output_details()[0]['index']
        self.history   = []
        self.total_inf = 0
        print(f"InferenceEngine prêt — {model_path}")

    def preprocess(self, image):
        img = tf.image.resize(image, [self.img_size, self.img_size]).numpy()
        return np.expand_dims(img.astype(np.float32) / 255.0, 0)

    def predict(self, image):
        tensor = self.preprocess(image)
        t0 = time.perf_counter()
        self.interpreter.set_tensor(self.inp_idx, tensor)
        self.interpreter.invoke()
        prob = float(self.interpreter.get_tensor(self.outp_idx)[0][0])
        latency_ms = (time.perf_counter() - t0) * 1000
        label      = self.labels[int(prob > 0.5)]
        confidence = prob if prob > 0.5 else 1 - prob
        result = {'label': label, 'confidence': confidence,
                  'latency_ms': latency_ms, 'raw_prob': prob}
        self.history.append(result)
        self.total_inf += 1
        return result

    def get_stats(self):
        if not self.history: return {}
        lats = [r['latency_ms'] for r in self.history]
        return {
            'total'         : self.total_inf,
            'avg_latency_ms': round(np.mean(lats), 2),
            'p95_latency_ms': round(np.percentile(lats, 95), 2),
            'fps'           : round(1000 / np.mean(lats), 1),
            'avg_confidence': round(np.mean([r['confidence'] for r in self.history]), 3),
        }

print("Classe InferenceEngine définie ✓")


In [ ]:
engine = InferenceEngine('model_int8_dynamic.tflite')

for i, img in enumerate(test_imgs[:50]):
    r   = engine.predict(img)
    bar = "#" * int(r['confidence'] * 20)
    print(f"[{i+1:02d}] {r['label']:<6} | {r['confidence']:.1%} [{bar:<20}] {r['latency_ms']:.1f}ms")

print("\n── Statistiques globales ──")
for k, v in engine.get_stats().items():
    print(f"  {k:<18} : {v}")

In [ ]:
def compute_score(results, w_acc=0.5, w_speed=0.3, w_size=0.2):
    accs  = np.array([r['accuracy']        for r in results.values()])
    lats  = np.array([r['latency_mean_ms'] for r in results.values()])
    sizes = np.array([r['size_mb']         for r in results.values()])
    acc_n   = (accs  - accs.min())  / (accs.max()  - accs.min()  + 1e-8)
    spd_n   = 1 - (lats  - lats.min())  / (lats.max()  - lats.min()  + 1e-8)
    siz_n   = 1 - (sizes - sizes.min()) / (sizes.max() - sizes.min() + 1e-8)
    scores  = w_acc * acc_n + w_speed * spd_n + w_size * siz_n
    print(f"\nProfil (précision={w_acc}, vitesse={w_speed}, taille={w_size})")
    print("─" * 50)
    for name, score in zip(results.keys(), scores):
        print(f"  {name:<18} {score:.3f} {'█'*int(score*20)}")
    best = list(results.keys())[np.argmax(scores)]
    print(f"  → Recommandation : {best}")

compute_score(results, w_acc=0.7, w_speed=0.2, w_size=0.1)  # précision critique
compute_score(results, w_acc=0.3, w_speed=0.5, w_size=0.2)  # temps réel
compute_score(results, w_acc=0.2, w_speed=0.3, w_size=0.5)  # mémoire limitée

#partie de deploiment 2

In [ ]:
!pip install gradio --quiet


In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
import time
from PIL import Image

# ── Chargement de l'interpréteur TFLite (une seule fois)
interpreter = tf.lite.Interpreter(model_path='model_int8_dynamic.tflite')
interpreter.allocate_tensors()
inp_idx  = interpreter.get_input_details()[0]['index']
outp_idx = interpreter.get_output_details()[0]['index']

def predict_image(image):
    """
    Reçoit une image PIL depuis Gradio,
    retourne un dictionnaire de scores pour l'affichage.
    """
    if image is None:
        return {"Chat": 0.5, "Chien": 0.5}

    # ── Prétraitement : redimensionne et normalise
    img = np.array(image.resize((160, 160))).astype(np.float32) / 255.0
    tensor = np.expand_dims(img, axis=0)   # (1, 160, 160, 3)

    # ── Inférence TFLite
    t0 = time.perf_counter()
    interpreter.set_tensor(inp_idx, tensor)
    interpreter.invoke()
    prob = float(interpreter.get_tensor(outp_idx)[0][0])
    latency_ms = (time.perf_counter() - t0) * 1000

    # ── Résultat formaté pour Gradio Label
    chat_score  = round((1 - prob) * 100, 1)
    chien_score = round(prob * 100, 1)
    label       = "Chien" if prob > 0.5 else "Chat"
    confidence  = chien_score if prob > 0.5 else chat_score

    print(f"[LOG] {label} — confiance : {confidence}% — latence : {latency_ms:.1f}ms")

    return {
        "Chien": float(prob),
        "Chat" : float(1 - prob),
    }

print("Fonction predict_image prête ✓")

In [ ]:
interface = gr.Interface(
    fn=predict_image,

    inputs=gr.Image(
        type="pil",          # reçoit l'image comme objet PIL
        label="Ta photo",
    ),

    outputs=gr.Label(
        num_top_classes=2,   # affiche Chat ET Chien avec leurs scores
        label="Résultat",
    ),

    title="Classificateur Chats vs Chiens",
    description=(
        "Upload une photo de chat ou de chien. "
        "Le modèle MobileNetV2 + TFLite int8 prédit la classe en temps réel."
    ),

    examples=[],             # tu peux ajouter des chemins d'images ici plus tard

    theme=gr.themes.Soft(),  # thème propre et lisible
)

interface.launch(share=True)   # share=True génère le lien public